# T-Rex — exhaustive checkpoint evaluation

Runs all three of T-Rex's own eval scripts against one checkpoint, plus the
attention-capture pass (now including the tactile-to-latent/tactile-to-action
maps added alongside this notebook), and writes one consolidated
`evaluation_summary.md` into your output directory as it goes -- not just
printed cell output, so the results survive even if the session dies or the
tab closes partway through.

1. **`eval_origami.py`** -- error, latency, safety (same script `eval_trex.ipynb`
   and `trex_colab.ipynb` use).
2. **`eval_diagnostics.py`** -- bias/variance split, anchored horizon, per-joint
   beat-count, chunk smoothness.
3. **`eval_robustness.py`** -- executed-prefix error, receding-horizon seams,
   delay sensitivity, tactile/sensor dropout. Run on the **pilot (stride-5)**
   val split even if the checkpoint is full-tier -- one dataset row is one
   delay quantum, so stride-20 can't resolve the sub-second sweep this
   actually tests (`trex_colab.ipynb` §8's own reasoning).
4. **Attention maps** -- same `run_validation(capture_attention=True)`
   mechanism as `trex_eval_analysis.ipynb`'s §6, featuring the two maps that
   turned out to carry real signal (`action_to_latent`, and now
   `tactile_to_latent`/`tactile_to_action`) and summarizing the rest
   numerically instead of as full heatmap dumps -- see §7's markdown for why.

Checkpoint loading is identical to `eval_trex.ipynb`/`trex_eval_analysis.ipynb`
(same three-way CHECKPOINT_ZIP/CHECKPOINT_DIR/CHECKPOINT_NAME resolution), and
the Pillow/accelerator fixes fought for in that session are baked in here
directly instead of being rediscovered.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv


## 1. Setup -- code + deps


In [ ]:
import os

PROJECT_ROOT = "/content/origami_trex/T-Rex"

if not os.path.isdir(PROJECT_ROOT):
    token = ""
    try:
        from google.colab import userdata
        token = userdata.get("GITHUB_PAT") or ""
    except Exception:
        pass
    token = token or os.environ.get("GITHUB_PAT", "")
    if not token:
        print("No GITHUB_PAT secret/env var found. Either add one (Colab "
              "secrets icon) or clone manually with the token from "
              "trex_colab.ipynb's own clone cell, then re-run this cell.")
    auth = f"{token}@" if token else ""
    !git clone https://{auth}github.com/KshitijBhat/origami_trex.git /content/origami_trex
else:
    print(f"already present: {PROJECT_ROOT}")


In [ ]:
%%bash
command -v uv >/dev/null 2>&1 || pip install -q uv
UV_PY="$(command -v python)"
uv pip install --python "${UV_PY}" -q \
    torch==2.6.0 torchvision==0.21.0 --index-url https://download.pytorch.org/whl/cu124
uv pip install --python "${UV_PY}" -q -e /content/origami_trex/T-Rex
uv pip install --python "${UV_PY}" -q bitsandbytes


This Colab image's preinstalled Pillow has been unreliable in prior sessions
-- `PIL.ImageText` importing `_Ink` from `PIL._typing`, which doesn't define
it, surfacing the moment `transformers`/`torchvision` get imported. Pinning to
the version that matches the compiled `_imaging` extension fixed it last time.
Cheap and idempotent, so it just always runs here rather than waiting for the
error.

**If this cell actually changes anything (prints "Successfully installed"),
you MUST do Runtime -> Restart session before continuing** -- a plain re-run
doesn't reload an already-imported C-extension package, which is exactly what
cost an entire prior debugging session (chased three different Pillow
versions before realizing the kernel simply hadn't restarted).


In [ ]:
!pip install -q --force-reinstall "Pillow==11.3.0"
import PIL
print("PIL version:", PIL.__version__, "-- if a restart just happened, this should read 11.3.0")


## 2. Checkpoint, data, and output parameters

Checkpoint resolution: same three-way CHECKPOINT_ZIP/CHECKPOINT_DIR/CHECKPOINT_NAME
convention as `trex_eval_analysis.ipynb` -- see that notebook's §2 for the
full explanation of the three ways to point this at a checkpoint.

**`OUTPUT_DIR` is the one thing you need to set.** Everything below writes
into subdirectories of it: `accuracy/`, `diagnostics/`, `robustness/`,
`attention/`, plus `evaluation_summary.md` at the top level.

The per-script sample counts default to `trex_colab.ipynb` §8's own "full
checkpoint, every eval" values -- lower them for a faster pass, e.g. while
iterating on this notebook itself.


In [ ]:
CHECKPOINT_ZIP = ""                     # only if you have a zip, not a dir
CHECKPOINT_DIR = ""                     # exact path if you know it
CHECKPOINT_NAME = "checkpoint-0-5"      # used to auto-locate only when CHECKPOINT_DIR is left empty
BASE_MODEL_PATH = ""   # only needed if the checkpoint's own config.json is missing

VAL_ROOT = "/content/data/origami_flat/full/val"             # accuracy + diagnostics
ROBUSTNESS_VAL_ROOT = "/content/data/origami_flat/pilot/val"  # stride-5 -- see §0 above
URDF_PATH = "/content/urdf/north_poc2_2_v3_1.urdf"

OUTPUT_DIR = ""   # <-- SET THIS. e.g. "/content/drive/MyDrive/iros2026/exhaustive_eval/<run-name>"
assert OUTPUT_DIR, "set OUTPUT_DIR before running the rest of this notebook"

EVAL_BATCH_SIZE = 8
NUM_WORKERS = 2             # 0 rules out Drive-FUSE / pyarrow-fork dataloader
                            # weirdness in Colab if a run seems to hang

# --- eval_origami.py (accuracy/latency/safety) ---
NUM_EVAL_SAMPLES = 2000
TRACE_EPISODES = 10
LATENCY_SAMPLES = 20

# --- eval_diagnostics.py (bias/variance, anchored horizon, smoothness) ---
DIAG_EPISODES = 10
DIAG_ROWS_PER_EPISODE = 60
DIAG_N_SAMPLES = 8          # flow redraws per row -- ~8x cost/sample vs eval_origami

# --- eval_robustness.py (executed-prefix, seams, delay sensitivity, dropout) ---
ROBUSTNESS_SAMPLES = 400
STITCH_ROWS = 240

# --- attention capture ---
ATTN_NUM_EPISODES = 2      # separate, small -- attention capture only looks at one batch anyway
ATTN_N_SAMPLES = 4

ACCURACY_DIR    = os.path.join(OUTPUT_DIR, "accuracy")
DIAGNOSTICS_DIR = os.path.join(OUTPUT_DIR, "diagnostics")
ROBUSTNESS_DIR  = os.path.join(OUTPUT_DIR, "robustness")
ATTN_VIZ_DIR    = os.path.join(OUTPUT_DIR, "attention")
LOG_PATH        = os.path.join(OUTPUT_DIR, "evaluation_summary.md")
for d in (OUTPUT_DIR, ACCURACY_DIR, DIAGNOSTICS_DIR, ROBUSTNESS_DIR, ATTN_VIZ_DIR):
    os.makedirs(d, exist_ok=True)


In [ ]:
import glob

if CHECKPOINT_ZIP:
    CHECKPOINT_DIR = CHECKPOINT_DIR or "/content/checkpoint"
    if not glob.glob(os.path.join(CHECKPOINT_DIR, "**", "model.pt"), recursive=True):
        os.makedirs(CHECKPOINT_DIR, exist_ok=True)
        print(f"unzipping {CHECKPOINT_ZIP} -> {CHECKPOINT_DIR}")
        !unzip -q -o "{CHECKPOINT_ZIP}" -d "{CHECKPOINT_DIR}"
elif not CHECKPOINT_DIR:
    print(f"searching Drive for a '{CHECKPOINT_NAME}' folder (this can take a bit)...")
    candidates = glob.glob(f"/content/drive/MyDrive/**/{CHECKPOINT_NAME}", recursive=True)
    candidates = [c for c in candidates if os.path.isdir(c)
                 and glob.glob(os.path.join(c, "**", "model.pt"), recursive=True)]
    assert candidates, (f"no folder named '{CHECKPOINT_NAME}' with a model.pt inside it "
                        f"found under /content/drive/MyDrive -- double check the name, or "
                        f"just set CHECKPOINT_DIR directly instead.")
    if len(candidates) > 1:
        print(f"WARNING: {len(candidates)} matches, using the first -- set "
              f"CHECKPOINT_DIR explicitly if this isn't the one you want:")
        for c in candidates:
            print(" ", c)
    CHECKPOINT_DIR = candidates[0]

hits = glob.glob(os.path.join(CHECKPOINT_DIR, "**", "model.pt"), recursive=True)
assert hits, f"no model.pt under {CHECKPOINT_DIR} -- check CHECKPOINT_ZIP/CHECKPOINT_DIR"
CHECKPOINT_DIR = os.path.dirname(hits[0])
print("CHECKPOINT_DIR ->", CHECKPOINT_DIR)
print("  processor/:", os.path.isdir(os.path.join(CHECKPOINT_DIR, "processor")))
print("  config.json:", os.path.exists(os.path.join(CHECKPOINT_DIR, "config.json")))
print("  training_args.json:", os.path.exists(os.path.join(CHECKPOINT_DIR, "training_args.json")))


## 3. Consolidated log

One markdown file, appended to after each stage, so the full picture survives
independent of cell output. Overwritten fresh at the start of each notebook
run (re-running from here restarts the log, it doesn't append duplicate runs).


In [ ]:
import datetime, json as _json

def log_write(mode, text):
    with open(LOG_PATH, mode) as f:
        f.write(text)

def log_section(title, lines):
    log_write("a", f"\n## {title}\n\n" + "\n".join(lines) + "\n")
    print(f"[log] appended '{title}' -> {LOG_PATH}")

log_write("w", (
    f"# Exhaustive evaluation\n\n"
    f"- checkpoint: `{CHECKPOINT_DIR}`\n"
    f"- val_root (accuracy/diagnostics): `{VAL_ROOT}`\n"
    f"- val_root (robustness): `{ROBUSTNESS_VAL_ROOT}`\n"
    f"- generated: {datetime.datetime.now().isoformat(timespec='seconds')}\n"
))
print("log initialized ->", LOG_PATH)


## 4. `eval_origami.py` -- error, latency, safety

Runs via `subprocess` + a manual line-read loop, not `%%bash` -- `%%bash`'s own
output capture doesn't reliably stream live in every Jupyter/Colab combination
(some only flush once the whole subprocess exits, `PYTHONUNBUFFERED` or not).
Printing each line through the notebook kernel's own stdout sidesteps that
entirely.


In [ ]:
import subprocess, sys

def run_streamed(cmd, cwd="/content/origami_trex/T-Rex"):
    env = {**os.environ, "PYTHONPATH": "/content/origami_trex/T-Rex", "PYTHONUNBUFFERED": "1"}
    proc = subprocess.Popen(cmd, cwd=cwd, env=env, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end="")
    proc.wait()
    if proc.returncode != 0:
        raise RuntimeError(f"{cmd[1]} exited with code {proc.returncode}")


In [ ]:
run_streamed([
    sys.executable, "scripts/eval_origami.py",
    "--checkpoint_path", CHECKPOINT_DIR,
    "--origami_root", VAL_ROOT,
    "--out_dir", ACCURACY_DIR,
    "--urdf", URDF_PATH,
    "--num_eval_samples", str(NUM_EVAL_SAMPLES),
    "--batch_size", str(EVAL_BATCH_SIZE),
    "--modes", "cascaded", "blind",
    "--latency_samples", str(LATENCY_SAMPLES),
    "--trace_episodes", str(TRACE_EPISODES),
    "--num_workers", str(NUM_WORKERS),
])


In [ ]:
with open(os.path.join(ACCURACY_DIR, "metrics.json")) as f:
    accuracy_metrics = _json.load(f)

lines = [f"| mode | MAE (deg) | RMSE (deg) | MAE contact (deg) |",
         f"|---|---|---|---|"]
print(f"{'mode':<16} {'MAE(deg)':>10} {'RMSE(deg)':>10} {'MAE contact':>12}")
print("-" * 52)
for name, report in accuracy_metrics["results"].items():
    overall, contact = report.get("overall", {}), report.get("contact", {})
    print(f"{name:<16} {overall.get('mae_deg', float('nan')):>10.3f} "
          f"{overall.get('rmse_deg', float('nan')):>10.3f} "
          f"{contact.get('mae_deg', float('nan')):>12.3f}")
    lines.append(f"| {name} | {overall.get('mae_deg', float('nan')):.3f} | "
                 f"{overall.get('rmse_deg', float('nan')):.3f} | "
                 f"{contact.get('mae_deg', float('nan')):.3f} |")

lines.append("")
lines.append("| phase | mean (ms) | p50 (ms) | p95 (ms) |")
lines.append("|---|---|---|---|")
print()
print(f"{'phase':<20} {'mean(ms)':>10} {'p50(ms)':>10} {'p95(ms)':>10}")
print("-" * 52)
for phase, v in accuracy_metrics.get("request_latency_ms", {}).items():
    print(f"{phase:<20} {v['mean']:>10.1f} {v['p50']:>10.1f} {v['p95']:>10.1f}")
    lines.append(f"| {phase} | {v['mean']:.1f} | {v['p50']:.1f} | {v['p95']:.1f} |")

safety = accuracy_metrics.get("safety", {})
if safety:
    lines.append("")
    lines.append("Safety/URDF violation rates: " +
                 ", ".join(f"{k}={v}" for k, v in safety.items()))

log_section("1. eval_origami.py -- error, latency, safety", lines)


In [ ]:
from IPython.display import Image, display

png_order = ["parity_plot.png", "horizon_error.png", "group_error.png",
             "group_error_distribution.png", "episode_trace.png"]
png_order += sorted(f for f in os.listdir(ACCURACY_DIR)
                    if f.endswith(".png") and f not in png_order)
for name in png_order:
    path = os.path.join(ACCURACY_DIR, name)
    if os.path.exists(path):
        print(name)
        display(Image(filename=path))


## 5. `eval_diagnostics.py` -- bias/variance, anchored horizon, smoothness

`--n_samples 8` redraws the flow 8x per row to split sampling noise from
irreducible bias, so this is intentionally run on far fewer rows than §4.


In [ ]:
run_streamed([
    sys.executable, "scripts/eval_diagnostics.py",
    "--checkpoint_path", CHECKPOINT_DIR,
    "--origami_root", VAL_ROOT,
    "--out_dir", DIAGNOSTICS_DIR,
    "--mode", "cascaded",
    "--n_samples", str(DIAG_N_SAMPLES),
    "--episodes", str(DIAG_EPISODES),
    "--rows_per_episode", str(DIAG_ROWS_PER_EPISODE),
    "--batch_size", str(EVAL_BATCH_SIZE),
    "--num_workers", str(NUM_WORKERS),
])


In [ ]:
with open(os.path.join(DIAGNOSTICS_DIR, "diagnostics.json")) as f:
    diagnostics_metrics = _json.load(f)

lines = []
for key in ("removable_fraction_of_mse", "anchored", "smoothness"):
    if key in diagnostics_metrics:
        print(f"{key}: {diagnostics_metrics[key]}")
        lines.append(f"- **{key}**: {diagnostics_metrics[key]}")

per_joint_csv = os.path.join(DIAGNOSTICS_DIR, "per_joint_ratio.csv")
if os.path.exists(per_joint_csv):
    import csv
    with open(per_joint_csv) as f:
        rows = list(csv.DictReader(f))
    beat_col = next((c for c in (rows[0].keys() if rows else []) if "beat" in c.lower()), None)
    if beat_col:
        n_beat = sum(1 for r in rows if r[beat_col].strip().lower() in ("true", "1"))
        print(f"joints beating hold_state baseline: {n_beat}/{len(rows)}")
        lines.append(f"- **joints beating hold_state baseline**: {n_beat}/{len(rows)}")

for name in ("horizon_anchored.png", "bias_variance.png", "per_joint_ratio.png", "smoothness.png"):
    path = os.path.join(DIAGNOSTICS_DIR, name)
    if os.path.exists(path):
        print(name)
        display(Image(filename=path))

log_section("2. eval_diagnostics.py -- bias/variance, anchored horizon, smoothness", lines)


## 6. `eval_robustness.py` -- deployment robustness

Run against `ROBUSTNESS_VAL_ROOT` (pilot, stride-5) regardless of which split
the checkpoint was trained on -- see the note in §0/§2.


In [ ]:
run_streamed([
    sys.executable, "scripts/eval_robustness.py",
    "--checkpoint_path", CHECKPOINT_DIR,
    "--origami_root", ROBUSTNESS_VAL_ROOT,
    "--out_dir", ROBUSTNESS_DIR,
    "--urdf", URDF_PATH,
    "--robustness_samples", str(ROBUSTNESS_SAMPLES),
    "--stitch_rows", str(STITCH_ROWS),
    "--batch_size", str(EVAL_BATCH_SIZE),
    "--num_workers", str(NUM_WORKERS),
])


In [ ]:
with open(os.path.join(ROBUSTNESS_DIR, "robustness.json")) as f:
    robustness_metrics = _json.load(f)

lines = []
for key, val in robustness_metrics.items():
    if isinstance(val, (int, float, str)):
        print(f"{key}: {val}")
        lines.append(f"- **{key}**: {val}")

for name in ("delay_sensitivity.png", "executed_prefix.png", "condition_ranking.png", "receding_horizon.png"):
    path = os.path.join(ROBUSTNESS_DIR, name)
    if os.path.exists(path):
        print(name)
        display(Image(filename=path))

log_section("3. eval_robustness.py -- deployment robustness", lines)


## 7. Attention maps

Same `run_validation(capture_attention=True)` mechanism as
`trex_eval_analysis.ipynb`'s §6, now with the tactile capture added alongside
this notebook (`tactile_flow_continue(output_attentions=True)`, wired into
`train.py`'s `has_any_tac` branch as a viz-only addition -- see
`qwen_vla/attention_capture.py`'s `save_tactile_attention_maps`).

**Featured vs. summarized, from directly inspecting a real run's `.npy`
arrays earlier this session:**
- `action_to_latent` and (new) `tactile_to_latent` / `tactile_to_action` --
  full heatmaps, per sample. These are the ones that showed real,
  interpretable structure (specific "landmark" tokens the action/tactile
  experts consistently attend to).
- `latent_to_action` -- **always exactly zero** at every layer/sample, by
  causal-masking construction (latent tokens can't attend forward to action
  tokens). Not plotted; just asserted as a sanity check.
- `latent_to_latent`, `action_to_action`, `tactile_to_tactile`
  (self-attention blocks) -- dominated by an attention-sink at token index 0
  that visually drowns out finer structure. Summarized numerically (row-sum
  sanity check, sink-mass fraction) instead of plotted per sample.


In [ ]:
import sys, importlib.util
from types import SimpleNamespace

sys.path.insert(0, "/content/origami_trex/T-Rex/scripts")
sys.path.insert(0, "/content/origami_trex/T-Rex")

import torch

def _load_by_path(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod

test_mod = _load_by_path("trex_test_server", "/content/origami_trex/T-Rex/scripts/test.py")
eo = _load_by_path("trex_eval_origami", "/content/origami_trex/T-Rex/scripts/eval_origami.py")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

load_args = SimpleNamespace(
    checkpoint_path=CHECKPOINT_DIR, base_model_path=BASE_MODEL_PATH,
    stats_path="", dataset_name="",
    action_dim=65, action_chunk=25,
    use_robot_state=1, use_tactile_vec=1, use_tactile_deform=1,
    tactile_intermediate_size=0, n_flare_tokens_per_frame=0, n_flare_steps=0,
    use_tactile_code=0, vqvae_codebook_size=64, use_tactile_vqvae=0, vqvae_config=None,
    cascaded_total_steps=10, cascaded_split_step=6,
)
model, processor, statistic = test_mod.model_load(load_args)
model = model.to(device).eval()
print(f"model loaded: action_dim={load_args.action_dim} action_chunk={load_args.action_chunk} "
      f"n_flare_tokens={model.n_flare_tokens}")


In [ ]:
from torch.utils.data import DataLoader, Subset

attn_config = eo.dataset_config_from_checkpoint(CHECKPOINT_DIR, {})
attn_dataset = eo.OrigamiDataset(attn_config, processor, eo._Printer(), root=VAL_ROOT)

attn_n_rows = sum(attn_dataset.ep_rows[:ATTN_NUM_EPISODES])
attn_episode_subset = Subset(attn_dataset, list(range(attn_n_rows)))
print(f"first {ATTN_NUM_EPISODES} episodes -> {attn_n_rows} samples: "
      f"{[attn_dataset.episodes[i]['file'] for i in range(ATTN_NUM_EPISODES)]}")

attn_batch_size = min(ATTN_N_SAMPLES, attn_n_rows)
attn_loader = DataLoader(attn_episode_subset, batch_size=attn_batch_size, shuffle=False,
                         num_workers=0, collate_fn=attn_dataset.collate_fn)


In [ ]:
from accelerate import Accelerator

train_mod = _load_by_path("trex_train", "/content/origami_trex/T-Rex/scripts/train.py")
accelerator = Accelerator()

load_args.max_val_batches = 1

K = model.n_flare_tokens
use_flare = K > 0

# run_validation assumes accelerate already placed batches on the right
# device (true in train.py's own main(), not true here -- see
# trex_eval_analysis.ipynb's §6 for the full explanation of why this line is
# required, not optional).
model, attn_loader = accelerator.prepare(model, attn_loader)

val_metrics = train_mod.run_validation(
    model, attn_loader, accelerator, load_args,
    is_stage1=False, use_flare=use_flare, K=K, T_per_frame=0, flare_layer_idx=0,
    capture_attention=True, viz_dir=ATTN_VIZ_DIR, capture_n_samples=attn_batch_size,
)
print(val_metrics)
print(f"saved to {ATTN_VIZ_DIR}:", sorted(os.listdir(ATTN_VIZ_DIR)))


In [ ]:
import numpy as np
import matplotlib.image as mpimg
import matplotlib.pyplot as plt

FEATURED = ("action_to_latent", "tactile_to_latent", "tactile_to_action")
SUMMARIZE_ONLY = ("latent_to_latent", "action_to_action", "tactile_to_tactile")
ALWAYS_ZERO = ("latent_to_action", "latent_to_tactile", "action_to_tactile")

all_npy = sorted(f for f in os.listdir(ATTN_VIZ_DIR) if f.endswith(".npy"))

def _combo(fname):
    # "{prefix}layer{N}_{q}_to_{k}.npy" -> "{q}_to_{k}"
    stem = fname[:-4]
    return "_".join(stem.split("_")[-3:]) if "_to_" in stem else stem

featured_pngs = sorted(
    f for f in os.listdir(ATTN_VIZ_DIR)
    if f.endswith(".png") and any(f"_{name}_sample" in f for name in FEATURED)
)
print(f"{len(featured_pngs)} featured heatmap(s):")
if featured_pngs:
    ncols = 4
    nrows = (len(featured_pngs) + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 4 * nrows))
    axes = axes.flatten() if len(featured_pngs) > 1 else [axes]
    for ax, name in zip(axes, featured_pngs):
        ax.imshow(mpimg.imread(os.path.join(ATTN_VIZ_DIR, name)))
        ax.set_title(name.replace(".png", ""), fontsize=7)
        ax.axis("off")
    for ax in axes[len(featured_pngs):]:
        ax.axis("off")
    fig.tight_layout()
    plt.show()
else:
    print("none saved -- see the [viz] warning printed above (e.g. no tactile "
          "data in this batch, or capture failed and validation continued anyway).")

summary_lines = [f"Featured heatmaps saved: {len(featured_pngs)} (see {ATTN_VIZ_DIR})", ""]
for f in all_npy:
    combo = _combo(f)
    if combo in ALWAYS_ZERO:
        arr = np.load(os.path.join(ATTN_VIZ_DIR, f))
        status = "OK (all zero, as expected -- causal masking)" if arr.max() == 0 else "UNEXPECTED NON-ZERO -- investigate"
        print(f"{f}: {status}")
        summary_lines.append(f"- `{f}`: {status}")
    elif combo in SUMMARIZE_ONLY:
        arr = np.load(os.path.join(ATTN_VIZ_DIR, f))
        rowsum = arr.sum(axis=-1)
        sink_frac = float(arr[..., 0].sum() / max(arr.sum(), 1e-9))
        line = (f"rowsum in [{rowsum.min():.3f}, {rowsum.max():.3f}], "
                f"sink-mass (key index 0) fraction={sink_frac:.3f}")
        print(f"{f}: {line}")
        summary_lines.append(f"- `{f}`: {line}")

log_section("4. Attention maps", summary_lines)


In [ ]:
print("=" * 60)
print(f"All done. Consolidated log: {LOG_PATH}")
print("=" * 60)
with open(LOG_PATH) as f:
    print(f.read())
